In [1]:
# Libraries
import pandas as pd
import numpy as np
# Working directory
home_dir = (
    "/Users/tim/Library/CloudStorage/"
    + "OneDrive-TheUniversityofColoradoDenver/Vigers/CF/Kamyron Jordan/"
    + "Racial and Ethnic Differences in ETI"
)

In [ ]:
# Import encounter data and demographics
encounters = pd.read_csv(
    home_dir + "/Data_Raw/DataDelivery_20240125/CFF22_encountersMerged_Del1.csv"
)
demo = pd.read_csv(
    home_dir + "/Data_Raw/DataDelivery_20240125/CFF22_DemogCFDiag_Del1.csv"
)
# Combine race columns
race_map = {
    "Race1": "White",
    "Race2": "Black or African American",
    "Race3": "American Indian or Alaska Native",
    "Race4": "Asian",
    "Race5": "Native Hawaiian or Other Pacific Islander",
    "Race6": "Some other race",
}
race_cols = list(race_map.keys())


def assign_race(row):
    selected = [race_map[c] for c in race_cols if row[c] == 1]
    if len(selected) > 1:
        return "Multiracial"
    elif len(selected) == 1:
        return selected[0]
    else:
        return "Unknown"


demo["Race"] = demo.apply(assign_race, axis=1)
# If someone is Hispanic, categorize them that way. If they are missing this
# info, make them unknown
demo.loc[demo["Hispanicrace"] == 1, "Race"] = "Hispanic or Latino"
demo.loc[np.isnan(demo["Hispanicrace"]), "Race"] = "Mixed/Other/Unknown Race"
# Combine categories
race_combos = {
    "White": "White",
    "Black or African American": "Black",
    "Hispanic or Latino": "Hispanic",
    "Asian": "Other",
    "American Indian or Alaska Native": "Other",
    "Native Hawaiian or Other Pacific Islander": "Other",
    "Mixed/Other/Unknown Race": "Other",
    "Some other race": "Other",
    "Multiracial": "Other",
}
demo["Race"] = demo["Race"].replace(race_combos)
# Add Trikafta start date and race to encounters
encounters = encounters.set_index(["eDWID", "reviewyear"])
demo = demo.set_index("eDWID")
encounters = encounters.join(demo["Modulator_trikafta_first_date"])
# Make pre/post Trikafta column
encounters["encounterdate"] = pd.to_datetime(encounters["encounterdate"])
encounters["Modulator_trikafta_first_date"] = pd.to_datetime(
    encounters["Modulator_trikafta_first_date"]
)
encounters["Pre-/Post-Trikafta"] = (
    encounters["encounterdate"] >= encounters["Modulator_trikafta_first_date"]
)
# Clean the data and note exclusions at each step for a consort diagram
n_start=len(encounters.index.unique())
e_start=encounters.shape[0]
# 


210719

In [19]:
encounters[["encounterdate","Modulator_trikafta_first_date","Pre-/Post-Trikafta"]].to_csv("~/trikafta.csv")